# Bayesian Model Comparison: Predictability & Cognitive Load

**Author:** Dipon Konar

Two analyses here — first one is about whether word predictability affects reading times, second one looks at cognitive load and pupil size. Both use brms for Bayesian inference.

Data comes from repeated-measures psycholinguistics experiments so hierarchical models are needed (can't just ignore the by-subject structure).

In [ ]:
library(brms)
library(bridgesampling)
library(ggplot2)

options(mc.cores = parallel::detectCores())
set.seed(42)

---

## Part 1 — Does word predictability speed up reading?

The dataset has reading times from 100 participants reading sentences where a target word was either predictable (coded +1) or unpredictable (−1). Basic idea: predictable words should be read faster because the brain pre-activates likely upcoming words.

RT data is right-skewed so I'm using a lognormal likelihood. The model on log-scale is:

$$rt_i \sim \text{Lognormal}(\mu_i,\ \sigma), \quad \mu_i = \alpha + \beta \cdot \text{pred}_i$$

Null model is the same but with $\beta = 0$. Priors:
- $\alpha \sim N(6, 1.5)$ — exp(6) ≈ 400ms which is in the right ballpark for reading
- $\beta \sim N(0, 1)$ — weakly informative, centered at zero
- $\sigma \sim N(0, 1)$

In [ ]:
pred_dat <- read.table("Predictibility-effect-data.csv", sep = ",", header = TRUE)[, -1]
pred_dat$rt <- round(pred_dat$rt, 2)

head(pred_dat)

### Bayes Factor Analysis

I'm using bridge sampling to get marginal likelihoods for each model and then computing BF. The BF tells us how many times more likely the data is under the predictability model vs the null.

One thing I wanted to check: does the BF change a lot depending on how wide the prior on $\beta$ is? Testing SD = 0.01, 0.1, 0.5, 5.

In [ ]:
# reusable prior builders so I don't keep copy-pasting
make_null_priors <- function() {
  c(
    set_prior("normal(6, 1.5)", class = "Intercept"),
    set_prior("normal(0, 1)",   class = "sigma")
  )
}

make_pred_priors <- function(beta_sd) {
  c(
    set_prior("normal(6, 1.5)",                       class = "Intercept"),
    set_prior(paste0("normal(0, ", beta_sd, ")"),     class = "b", coef = "pred"),
    set_prior("normal(0, 1)",                         class = "sigma")
  )
}

In [ ]:
# fit null once — reuse for all comparisons
m_null <- brm(
  rt ~ 1,
  data      = pred_dat,
  family    = lognormal(),
  prior     = make_null_priors(),
  cores     = 4, iter = 20000, warmup = 2000,
  save_pars = save_pars(all = TRUE)
)

margLogLik_null <- bridge_sampler(m_null, silent = TRUE)

In [ ]:
prior_sds  <- c(0.01, 0.1, 0.5, 5)
bf_results <- data.frame(prior_sd = prior_sds, bayes_factor = NA_real_)

for (i in seq_along(prior_sds)) {
  m_pred <- brm(
    rt ~ 1 + pred,
    data      = pred_dat,
    family    = lognormal(),
    prior     = make_pred_priors(prior_sds[i]),
    cores     = 4, iter = 20000, warmup = 2000,
    save_pars = save_pars(all = TRUE)
  )
  ml_pred <- bridge_sampler(m_pred, silent = TRUE)
  bf <- bayes_factor(ml_pred, margLogLik_null)
  bf_results$bayes_factor[i] <- bf$bf
  cat(sprintf("prior SD = %.2f  ->  BF = %.3g\n", prior_sds[i], bf$bf))
}

print(bf_results)

In [ ]:
ggplot(bf_results, aes(x = prior_sd, y = bayes_factor)) +
  geom_line(color = "steelblue", linewidth = 0.9) +
  geom_point(color = "steelblue", size = 3) +
  scale_y_log10() +
  scale_x_log10() +
  geom_hline(yintercept = 10, linetype = "dashed", color = "firebrick") +
  annotate("text", x = 0.015, y = 14, label = "BF = 10", color = "firebrick", size = 3.5) +
  labs(
    title = "Bayes Factor vs Prior Width on beta",
    x = "Prior SD (log scale)",
    y = "Bayes Factor (log scale)"
  ) +
  theme_minimal(base_size = 12)

BF is large for most priors — strong evidence for the predictability effect. The SD = 0.01 case basically tells the model that beta has to be nearly zero before seeing data, so the BF is low there, but that's not a realistic prior anyway.

### Cross-validation (k = 6)

Just to triangulate — does the predictability model also generalize better to held-out data? Using k-fold CV instead of BF here.

In [ ]:
m_pred_cv <- brm(
  rt ~ 1 + pred,
  data   = pred_dat, family = lognormal(),
  prior  = make_pred_priors(1), cores = 4
)

m_null_cv <- brm(
  rt ~ 1,
  data   = pred_dat, family = lognormal(),
  prior  = make_null_priors(), cores = 4
)

kfold_pred <- kfold(m_pred_cv, K = 6)
kfold_null <- kfold(m_null_cv, K = 6)

loo_compare(kfold_pred, kfold_null)

---

## Part 2 — Cognitive load and pupil size

Data from Wahn et al. (2016). Pupil size was measured across trials with varying cognitive load levels. Expectation is that higher load → larger pupils (task-evoked pupillary response).

I centred cognitive load so the intercept is interpretable as average pupil size at mean load.

Since there are multiple trials per subject, and subjects probably differ in baseline pupil size and how sensitive they are to load, I'm fitting a hierarchical model with by-subject varying intercepts and slopes (and letting them correlate).

Model:
$$p\_size_i \sim N(\mu_i, \sigma), \quad \mu_i = \alpha_j + \beta_j \cdot \text{c\_load}_i$$

Subject-level intercepts and slopes come from a bivariate normal with population means $\alpha$, $\beta$ and a covariance structure estimated from data.

Priors: $\alpha \sim N(1000, 500)$, $\beta \sim N(0, 100)$, $\sigma \sim N^+(0, 500)$, $\tau \sim N^+(0, 250)$, $\rho \sim LKJ(2)$

In [ ]:
dat <- read.csv("df_pupil_complete.csv")
dat$c_load <- dat$load - mean(dat$load)

cat("subjects:", length(unique(dat$subj)), "| observations:", nrow(dat), "\n")
summary(dat[, c("c_load", "p_size")])

In [ ]:
priors_hier <- c(
  set_prior("normal(1000, 500)", class = "Intercept"),
  set_prior("normal(0, 500)",    class = "sigma"),
  set_prior("normal(0, 100)",    class = "b"),
  set_prior("normal(0, 250)",    class = "sd"),
  set_prior("lkj(2)",            class = "cor")
)

m_hier <- brm(
  p_size ~ 1 + c_load + (1 + c_load | subj),
  data   = dat,
  prior  = priors_hier,
  chains = 2, cores = 2, iter = 3000, warmup = 1000
)

summary(m_hier)

In [ ]:
# effect of load with 95% CI
round(fixef(m_hier)["c_load", c("Estimate", "Q2.5", "Q97.5")], 2)

In [ ]:
# plot individual subject fits vs population average
ind <- coef(m_hier)$subj
x_seq <- seq(min(dat$c_load), max(dat$c_load), length.out = 80)

lines_df <- do.call(rbind, lapply(seq_len(dim(ind)[1]), function(i) {
  data.frame(x = x_seq,
             y = ind[i, 1, "Intercept"] + ind[i, 1, "c_load"] * x_seq,
             subj = i)
}))

pop_df <- data.frame(
  x = x_seq,
  y = fixef(m_hier)["Intercept", "Estimate"] + fixef(m_hier)["c_load", "Estimate"] * x_seq
)

ggplot() +
  geom_line(data = lines_df, aes(x = x, y = y, group = subj),
            color = "steelblue", alpha = 0.3, linewidth = 0.4) +
  geom_line(data = pop_df, aes(x = x, y = y),
            color = "firebrick", linewidth = 1.3) +
  labs(
    title = "Subject-level fits (blue) vs population average (red)",
    x = "Centered Cognitive Load", y = "Pupil Size"
  ) +
  theme_minimal(base_size = 12)

There's quite a bit of between-subject variability — some subjects show a much stronger load effect than others. The population average (red line) is positive.

### Bayes factors for hierarchical model

The CI above doesn't tell us whether there's actual evidence for an effect. For that, I need to do model comparison. Null model here keeps the random effects structure (removing them would be wrong — we know individuals differ) but drops the population-level $\beta$.

Testing several prior widths on $\beta$ to check robustness.

In [ ]:
priors_null_hier <- c(
  set_prior("normal(1000, 500)", class = "Intercept"),
  set_prior("normal(0, 500)",    class = "sigma"),
  set_prior("normal(0, 250)",    class = "sd"),
  set_prior("lkj(2)",            class = "cor")
)

mfit_null <- brm(
  p_size ~ 1 + (1 + c_load | subj),
  data      = dat, prior = priors_null_hier,
  chains    = 2, cores = 2, iter = 3000, warmup = 1000,
  save_pars = save_pars(all = TRUE)
)

ml_null_hier <- bridge_sampler(mfit_null, silent = TRUE)

In [ ]:
prior_sds_hier  <- c(5, 25, 50, 100, 200, 500)
bf_hier <- data.frame(prior_sd = prior_sds_hier, bayes_factor = NA_real_)

for (k in seq_along(prior_sds_hier)) {
  p <- prior_sds_hier[k]
  priors_full <- c(
    set_prior("normal(1000, 500)",               class = "Intercept"),
    set_prior(paste0("normal(0, ", p, ")"),      class = "b"),
    set_prior("normal(0, 500)",                  class = "sigma"),
    set_prior("normal(0, 250)",                  class = "sd"),
    set_prior("lkj(2)",                          class = "cor")
  )
  mfit_full <- brm(
    p_size ~ 1 + c_load + (1 + c_load | subj),
    data      = dat, prior = priors_full,
    chains    = 2, cores = 2, iter = 3000, warmup = 1000,
    save_pars = save_pars(all = TRUE)
  )
  ml_full <- bridge_sampler(mfit_full, silent = TRUE)
  bf <- bayes_factor(ml_full, ml_null_hier)
  bf_hier$bayes_factor[k] <- bf$bf
  cat(sprintf("prior SD = %d  ->  BF = %.3g\n", p, bf$bf))
}

print(bf_hier)

In [ ]:
ggplot(bf_hier, aes(x = prior_sd, y = bayes_factor)) +
  geom_line(color = "#2ca25f", linewidth = 0.9) +
  geom_point(color = "#2ca25f", size = 3) +
  scale_y_log10() +
  geom_hline(yintercept = 10, linetype = "dashed", color = "firebrick") +
  geom_hline(yintercept = 3,  linetype = "dotted",  color = "darkorange") +
  annotate("text", x = 20, y = 13,  label = "BF = 10", color = "firebrick",  size = 3.3) +
  annotate("text", x = 20, y = 3.8, label = "BF = 3",  color = "darkorange", size = 3.3) +
  labs(
    title = "Prior Sensitivity: Hierarchical BFs (cognitive load model)",
    x = "Prior SD on beta", y = "Bayes Factor (log scale)"
  ) +
  theme_minimal(base_size = 12)

---

## Summary

**Predictability and reading times:** BFs were consistently high across prior choices (except the degenerate SD = 0.01 case). K-fold CV agreed. So there's pretty strong evidence that predictable words are read faster.

**Cognitive load and pupil size:** Positive population-level effect, credible interval doesn't include zero. BFs drop as the prior gets wider — that's expected because broader priors put mass on effect sizes the data doesn't support. For priors with SD in the 50–100 range (which seem reasonable for pupil size units), evidence is still above BF = 10.

The prior sensitivity analysis is important here. If results only hold for one specific prior, that's a problem. Here both analyses are fairly robust across a range of reasonable prior choices, which I think is the main takeaway.